In [1]:
# 1. Tải mã nguồn PaddleOCR và di chuyển vào thư mục
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR

# 2. Cài đặt thư viện lõi (Bản có hỗ trợ GPU)
!pip install paddlepaddle-gpu -U
!pip install -r requirements.txt

%cd ..

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 337229, done.
remote: Counting objects: 100% (860/860), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 337229 (delta 716), reused 626 (delta 626), pack-reused 336369 (from 3)
Receiving objects: 100% (337229/337229), 1.79 GiB | 32.30 MiB/s, done.
Resolving deltas: 100% (267107/267107), done.
/kaggle/working/PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.9/758.9 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 36.7 kB/s eta 0:00:00ta 0:00:01
  Attempting uninstall: opt-einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0
Ignoring lmdb: markers 'python_version < "3.9"' don't match your environment
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 7.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 63.3 MB/s eta 0:00:00:00:01
/kaggle

In [2]:
# Tạo file từ điển (Bảng chữ cái biển số)
alphabet = "0123456789ABCDEFGHKLMNPQRSTUVXYZĐ"

dict_path = "/kaggle/working/PaddleOCR/ppocr/utils/plate_dict.txt"
with open(dict_path, "w", encoding="utf-8") as f:
    for char in alphabet:
        f.write(char + "\n")
print("Done")

Done


In [3]:
%%writefile /kaggle/working/PaddleOCR/configs/rec/rec_r34_vd_none_bilstm_ctc.yml
Global:
  use_gpu: true
  epoch_num: 500
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: ./output/rec/r34_vd_none_bilstm_ctc/
  save_epoch_step: 25
  # evaluation is run every 2000 iterations
  eval_batch_step: [0, 2000]
  cal_metric_during_train: True
  pretrained_model:
  checkpoints:
  save_inference_dir:
  use_visualdl: False
  infer_img: doc/imgs_words_en/word_10.png
  # for data or label process
  character_dict_path: /kaggle/working/PaddleOCR/ppocr/utils/plate_dict.txt
  max_text_length: 11
  infer_mode: False
  use_space_char: False
  save_res_path: ./output/rec/predicts_r34_vd_none_bilstm_ctc.txt

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    learning_rate: 0.0005
  regularizer:
    name: 'L2'
    factor: 0

Architecture:
  model_type: rec
  algorithm: CRNN
  Transform:
  Backbone:
    name: ResNet
    layers: 34
  Neck:
    name: SequenceEncoder
    encoder_type: rnn
    hidden_size: 256
  Head:
    name: CTCHead
    fc_decay: 0

Loss:
  name: CTCLoss

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc

Train:
  dataset:
    name: SimpleDataSet
    data_dir: /kaggle/input/datasets/lucpac/plate-vietnam/dataset_crnn_v5/train/images
    label_file_list: ["/kaggle/input/datasets/lucpac/plate-vietnam/dataset_crnn_v5/train/labels.txt"]
    transforms:
      - DecodeImage: # load image
          img_mode: BGR
          channel_first: False
      - CTCLabelEncode: # Class handling label
      - RecResizeImg:
          image_shape: [3, 32, 100]
      - KeepKeys:
          keep_keys: ['image', 'label', 'length'] # dataloader will return list in this order
  loader:
    shuffle: True
    batch_size_per_card: 192
    drop_last: True
    num_workers: 8

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: /kaggle/input/datasets/lucpac/plate-vietnam/dataset_crnn_v5/val/images
    label_file_list: ["/kaggle/input/datasets/lucpac/plate-vietnam/dataset_crnn_v5/val/labels.txt"]
    transforms:
      - DecodeImage: # load image
          img_mode: BGR
          channel_first: False
      - CTCLabelEncode: # Class handling label
      - RecResizeImg:
          image_shape: [3, 32, 100]
      - KeepKeys:
          keep_keys: ['image', 'label', 'length'] # dataloader will return list in this order
  loader:
    shuffle: False
    drop_last: False
    batch_size_per_card: 128
    num_workers: 4

Overwriting /kaggle/working/PaddleOCR/configs/rec/rec_r34_vd_none_bilstm_ctc.yml


In [4]:
%cd /kaggle/working/PaddleOCR
!python tools/train.py -c configs/rec/rec_r34_vd_none_bilstm_ctc.yml

/kaggle/working/PaddleOCR
Skipping import of the encryption module.
[2026/04/24 06:34:21] ppocr INFO: Architecture : 
[2026/04/24 06:34:21] ppocr INFO:     Backbone : 
[2026/04/24 06:34:21] ppocr INFO:         layers : 34
[2026/04/24 06:34:21] ppocr INFO:         name : ResNet
[2026/04/24 06:34:21] ppocr INFO:     Head : 
[2026/04/24 06:34:21] ppocr INFO:         fc_decay : 0
[2026/04/24 06:34:21] ppocr INFO:         name : CTCHead
[2026/04/24 06:34:21] ppocr INFO:     Neck : 
[2026/04/24 06:34:21] ppocr INFO:         encoder_type : rnn
[2026/04/24 06:34:21] ppocr INFO:         hidden_size : 256
[2026/04/24 06:34:21] ppocr INFO:         name : SequenceEncoder
[2026/04/24 06:34:21] ppocr INFO:     Transform : None
[2026/04/24 06:34:21] ppocr INFO:     algorithm : CRNN
[2026/04/24 06:34:21] ppocr INFO:     model_type : rec
[2026/04/24 06:34:21] ppocr INFO: Eval : 
[2026/04/24 06:34:21] ppocr INFO:     dataset : 
[2026/04/24 06:34:21] ppocr INFO:         data_dir : /kaggle/input/datasets/

In [5]:
!pip uninstall paddlepaddle paddlepaddle-gpu -y
!python -m pip install paddlepaddle-gpu==3.0.0rc1 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/

Found existing installation: paddlepaddle-gpu 2.6.2
Uninstalling paddlepaddle-gpu-2.6.2:
  Successfully uninstalled paddlepaddle-gpu-2.6.2
Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu118/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 GB 1.5 MB/s eta 0:00:00:00:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 47.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 10.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.9/699.9 MB 2.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 3.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 5.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 8.4 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 4.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 5.6 MB/s e

In [6]:
%cd /kaggle/working/PaddleOCR
!python tools/train.py -c configs/rec/rec_r34_vd_none_bilstm_ctc.yml

/kaggle/working/PaddleOCR
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2026/04/24 06:42:19] ppocr INFO: Architecture : 
[2026/04/24 06:42:19] ppocr INFO:     Backbone : 
[2026/04/24 06:42:19] ppocr INFO:         layers : 34
[2026/04/24 06:42:19] ppocr INFO:         name : ResNet
[2026/04/24 06:42:19] ppocr INFO:     Head : 
[2026/04/24 06:42:19] ppocr INFO:         fc_decay : 0
[2026/04/24 06:42:19] ppocr INFO:         name : CTCHead
[2026/04/24 06:42:19] ppocr INFO:     Neck : 
[2026/04/24 06:42:19] ppocr INFO:         encoder_type : rnn
[2026/04/24 06:42:19] ppocr INFO:         hidden_size : 256
[2026/04/24 06:42:19] ppocr INFO:         name : SequenceEncoder
[2026/04/

In [7]:
!python tools/export_model.py \
    -c configs/rec/rec_r34_vd_none_bilstm_ctc.yml \
    -o Global.pretrained_model=./output/rec/r34_vd_none_bilstm_ctc/best_accuracy \
    Global.character_dict_path=./ppocr/utils/plate_dict.txt \
    Global.save_inference_dir=./output/export_model

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
W0424 13:52:54.515383 17982 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 11.8
W0424 13:52:54.518644 17982 gpu_resources.cc:164] device: 0, cuDNN Version: 8.9.
[2026/04/24 13:52:55] ppocr INFO: load pretrain successful from ./output/rec/r34_vd_none_bilstm_ctc/best_accuracy
[2026/04/24 13:52:55] ppocr INFO: Export inference config file to ./output/export_model/inference.yml
Skipping import of the encryption module
[2026/04/24 13:52:56] ppocr INFO: inference model is saved to ./output/export_model/inference


In [10]:
!pip install paddle2onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 38.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 86.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 372.8/372.8 kB 27.2 MB/s eta 0:00:00
  Attempting uninstall: onnx
    Found existing installation: onnx 1.20.1
    Uninstalling onnx-1.20.1:
      Successfully uninstalled onnx-1.20.1


In [11]:
!paddle2onnx \
    --model_dir ./output/export_model \
    --model_filename inference.json \
    --params_filename inference.pdiparams \
    --save_file ./output/export_model/model.onnx \
    --opset_version 12 \
    --enable_onnx_checker True

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[Paddle2ONNX] Start parsing the Paddle model file...
[Paddle2ONNX] Use opset_version = 12 for ONNX export.
[Paddle2ONNX] PaddlePaddle model is exported as ONNX format now.
2026-04-24 13:53:59 [INFO]	Try to perform constant folding on the ONNX model with Polygraphy.
[W] 'colored' module is not installed, will not use colors when logging. To enable colors, please install the 'colored' module: python3 -m pip install colored
[I] Module: 'onnxruntime' is required, but not installed. Attempting to install now.
[I] Running installation command: /usr/bin/python3 -m pip install onnxruntime>=1.10.0
[I] Folding Constants | Pass 1
[I] Module: 'onnx_graphsurgeon' is required, but n

In [12]:
%cd /kaggle/working
!zip -r full_kaggle_workspace.zip ./*

/kaggle/working
  adding: PaddleOCR/ (stored 0%)
  adding: PaddleOCR/docs/ (stored 0%)
  adding: PaddleOCR/docs/update/ (stored 0%)
  adding: PaddleOCR/docs/update/update.md (deflated 62%)
  adding: PaddleOCR/docs/update/upgrade_notes.md (deflated 47%)
  adding: PaddleOCR/docs/update/upgrade_notes.en.md (deflated 56%)
  adding: PaddleOCR/docs/update/update.en.md (deflated 67%)
  adding: PaddleOCR/docs/quick_start.md (deflated 60%)
  adding: PaddleOCR/docs/version3.x/ (stored 0%)
  adding: PaddleOCR/docs/version3.x/logging.en.md (deflated 51%)
  adding: PaddleOCR/docs/version3.x/doc2md.en.md (deflated 68%)
  adding: PaddleOCR/docs/version3.x/other_devices_support/ (stored 0%)
  adding: PaddleOCR/docs/version3.x/other_devices_support/multi_devices_use_guide.en.md (deflated 62%)
  adding: PaddleOCR/docs/version3.x/other_devices_support/multi_devices_use_guide.md (deflated 56%)
  adding: PaddleOCR/docs/version3.x/other_devices_support/paddlepaddle_install_NPU.en.md (deflated 56%)
  adding:

In [13]:
from IPython.display import FileLink
FileLink(r'full_kaggle_workspace.zip') 

/kaggle/working/full_kaggle_workspace.zip